# 71 - Q50+U20 coefficient evaluation, worker 0/2

Runs 80 of the fixed 160 held-out LIBERO-PRO position-perturbation identities. Each identity runs Q50+U20 with beta 0.25, 0.5, and 1.0. The periodic table also includes the exact matched stock VLA outcome already logged by notebook 68; stock is not rerun.

Candidates are ranked by standardized Q minus beta times standardized predicted future U20. The selected top 16 are blended with the ordinary Q-softmax weights. Current U20 is measured live at every planning boundary. All arms execute 10 actions and replan. Video, frames, and generated chunks are off.


In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

SHARD_COUNT = 2
SHARD_INDEX = 0
EPISODE_LIMIT = None  # set to 1 only for a three-rollout smoke test
CANDIDATE_BATCH_SIZE = 8  # lower only after a GPU-memory error
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_u20')
Q50_U20_CHECKPOINT_PATH = None

if Q50_U20_CHECKPOINT_PATH is None:
    matches = sorted(OUTPUT_ROOT.glob(
        'pcpcds-*/q50_u20_full/checkpoint_step_008000.pt'))
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one final Q50+U20 checkpoint; found ' +
            f'{len(matches)}: {[str(path) for path in matches]}. ' +
            'Set Q50_U20_CHECKPOINT_PATH explicitly.')
    Q50_U20_CHECKPOINT_PATH = matches[0]
else:
    Q50_U20_CHECKPOINT_PATH = Path(Q50_U20_CHECKPOINT_PATH)
    if not Q50_U20_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(Q50_U20_CHECKPOINT_PATH)

print({
    'shard': f'{SHARD_INDEX}/{SHARD_COUNT}',
    'identities': 80 if EPISODE_LIMIT is None else EPISODE_LIMIT,
    'new_rollouts': 240 if EPISODE_LIMIT is None else 3 * EPISODE_LIMIT,
    'betas': [0.25, 0.5, 1.0],
    'candidate_batch_size': CANDIDATE_BATCH_SIZE,
    'checkpoint': str(Q50_U20_CHECKPOINT_PATH),
    'periodic_print_every_complete_identities': 10,
    'historic_stock': 'reuse exact notebook-68 rows',
    'video_frames_generated_chunks': 'off',
})


In [ ]:
from pnp.qplanning_u20_eval_experiment import run_qplanning_u20_heldout_worker

report = run_qplanning_u20_heldout_worker(
    checkpoint_path=Q50_U20_CHECKPOINT_PATH,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    episode_limit=EPISODE_LIMIT,
    candidate_batch_size=CANDIDATE_BATCH_SIZE,
)
report
